In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedKFold, train_test_split

# 1. Garantie des variables nécessaires même si la cellule est exécutée isolément
if "RANDOM_STATE" not in globals():
  RANDOM_STATE = 42

if "X" not in globals() or "y" not in globals() or "df_clean" not in globals():
  DATA_PATH = "../data/Addis_Ababa_city_RTA.csv"
  df_tmp = pd.read_csv(DATA_PATH)
  df_tmp.columns = [c.strip() for c in df_tmp.columns]

  casualty_candidates = [
      c
      for c in df_tmp.columns
      if "casualt" in c.lower()
      and not any(
          bad in c.lower()
          for bad in ["class", "sever", "age", "sex", "fitness", "work"]
      )
  ]
  target_col = (
      casualty_candidates[0] if casualty_candidates else df_tmp.columns[-1]
  )
  leakage_cols = [
      c
      for c in df_tmp.columns
      if c != target_col
      and any(k in c.lower() for k in ["casualty", "fitness", "severity"])
  ]

  df_clean = (
      df_tmp.drop_duplicates()
      .drop(columns=leakage_cols, errors="ignore")
      .copy()
  )
  df_clean[target_col] = pd.to_numeric(df_clean[target_col], errors="coerce")
  df_clean = df_clean.dropna(subset=[target_col]).reset_index(drop=True)

  y = df_clean[target_col].astype(float)
  X = df_clean.drop(columns=[target_col])
else:
  df_clean = df_clean.reset_index(drop=True)
  X = X.reset_index(drop=True)
  y = y.reset_index(drop=True)

# 2. Mise de côté d'un ensemble de test (20%) utilisé UNE SEULE FOIS pour l'évaluation finale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

# 3. Split de référence imposé pour TH7 G1 : KFold 5 folds répété 3 fois sur X_train
cv_ref = RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)

# 4. Export du dataset nettoyé avec la colonne 'split_ref' ('train' / 'test')
dataset_clean_export = df_clean.copy()
dataset_clean_export["split_ref"] = np.where(
    dataset_clean_export.index.isin(X_train.index), "train", "test"
)
dataset_clean_export.to_csv("../data/dataset_clean.csv", index=False)

print("=== PHASE 3 — SPLIT DE RÉFÉRENCE & EXPORT ===")
print(
    f"Train (80%) : {X_train.shape[0]} lignes | Test isolé (20%) :"
    f" {X_test.shape[0]} lignes"
)
print(
    "Validation croisée de référence : RepeatedKFold (5 folds x 3 repeats ="
    f" {cv_ref.get_n_splits()} splits)"
)
print("Fichier 'dataset_clean.csv' sauvegardé avec la colonne 'split_ref'.")
print(dataset_clean_export["split_ref"].value_counts())

In [ ]:

print("========== DATA VERIFICATION ==========")

print("Dataset shape:", df_clean.shape)
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget column:")
print(target_col)

print("\nTarget statistics:")
print(y.describe())

print("\nTraining set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nCross-validation:")
print(cv_ref)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nData types:")
print(X.dtypes.value_counts())

print("\nMissing values in X:")
print(X.isna().sum().sum())

print("\nMissing values in y:")
print(y.isna().sum())
